### Libraries

In [17]:
# to install packages if not installed
# %pip install pandas
# %pip install seaborn
# %pip install statsmodels
# %pip install matplotlib
# %pip install numpy

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import numpy as np

pd.set_option('display.width', 500)

In [19]:
df = pd.read_excel("data/Data_FE.xlsx", sheet_name="25 Size and BEME portfolios")
df.head()

,Unnamed: 0,Average,Value,Weighted,Returns,--,Monthly,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25
0,Size,Small,Small,Small,Small,Small,2,2.00,2.00,2.00,...,4,4.00,4.00,4.00,4,Big,Big,Big,Big,Big
1,BE/ME,Low,2,3,4,High,Low,2.00,3.00,4.00,...,Low,2.00,3.00,4.00,High,Low,2,3,4,High
2,193601,26.94,15.49,21.82,14.56,33.28,16.43,12.49,9.49,10.27,...,2.13,6.87,8.92,8.35,6.17,2.55,6.36,8.62,14.84,16.29
3,193602,9.46,12.78,6.77,10.02,9,1.7,6.71,5.61,8.92,...,2.59,4.04,6.14,4.80,8.43,1.88,1.18,3.99,3.38,3.75
4,193603,9.46,1.38,5.56,3.01,1.24,-0.37,1.35,3.33,-1.11,...,-0.46,-0.40,3.19,1.52,-4.36,3.58,1.27,-1.82,1.26,-3.56


In [20]:
# Set rows 1 and 2 as MultiIndex header
df.columns = pd.MultiIndex.from_arrays([df.iloc[0], df.iloc[1]])
df = df.iloc[2:].reset_index(drop=True)

# Convert Date column to Year-Month
date_col = df.columns[0]

df[date_col] = pd.to_datetime(df[date_col].astype(int), format="%Y%m")

df = df.set_index(df.columns[0]) 
df = df.apply(pd.to_numeric)

df.head()

0              Small                                  2                              ...     4                              Big                          
1                Low      2      3      4   High    Low      2      3      4   High  ...   Low     2      3      4   High   Low     2     3      4   High
(Size, BE/ME)                                                                        ...                                                                 
1936-01-01     26.94  15.49  21.82  14.56  33.28  16.43  12.49   9.49  10.27  26.81  ...  2.13  6.87   8.92   8.35   6.17  2.55  6.36  8.62  14.84  16.29
1936-02-01      9.46  12.78   6.77  10.02   9.00   1.70   6.71   5.61   8.92   3.87  ...  2.59  4.04   6.14   4.80   8.43  1.88  1.18  3.99   3.38   3.75
1936-03-01      9.46   1.38   5.56   3.01   1.24  -0.37   1.35   3.33  -1.11   1.25  ... -0.46 -0.40   3.19   1.52  -4.36  3.58  1.27 -1.82   1.26  -3.56
1936-04-01    -28.60 -29.05 -12.37 -14.03 -21.72 -19.41 -13.35 -15.93 -16.43 -17.69  ... -8.31 -9.00 -11.90 -12.09 -12.46 -6.17 -8.26 -7.05 -10.57  -8.00
1936-05-01      1.81   8.62   1.89  10.60   5.90   5.21   4.59   7.57   5.84   6.97  ...  5.12  1.79   4.54   5.98   8.18  4.78  5.20  4.71   6.25   8.39

[5 rows x 25 columns]

### Fama-French Factors Data

In [21]:
FF_factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Fama-French factors")

# Convert Date column to Year-Month
date_col = FF_factors.columns[0] 

FF_factors[date_col] = pd.to_datetime(FF_factors[date_col].astype(int), format="%Y%m")
FF_factors = FF_factors.set_index(FF_factors.columns[0]) 
FF_factors = FF_factors.apply(pd.to_numeric)

FF_factors.head()

,Mkt-RF,SMB,HML,RF
Date,,,,
1936-01-01,6.60,6.43,10.09,0.01
1936-02-01,2.56,0.77,3.98,0.01
1936-03-01,0.92,1.10,-2.23,0.02
1936-04-01,-8.07,-6.81,-2.18,0.02
1936-05-01,5.01,0.81,2.69,0.02


### Macroeconomic Factors Data

In [22]:
macro_factors = pd.read_excel("data/Data_FE.xlsx", sheet_name="Macroeconomic factors")

# Convert Date column to Year-Month
date_col = macro_factors.columns[0] 

macro_factors[date_col] = pd.to_datetime(macro_factors[date_col].astype(int), format="%Y%m")
macro_factors = macro_factors.set_index(macro_factors.columns[0]) 
macro_factors = macro_factors.apply(pd.to_numeric)

macro_factors.head()

,Div_growth,DEF = LT Corp. - LT govt.,TERM = ST govt.-LT govt.
Date,,,
1936-01-01,0.001990,0.002660,0.005395
1936-02-01,0.003113,-0.002690,0.007990
1936-03-01,0.003308,-0.002406,0.010458
1936-04-01,0.001250,-0.000924,0.003359
1936-05-01,0.004499,-0.000049,0.003887


### Excess Returns

In [23]:
excess_returns = df.copy()

FF_factors = FF_factors.reindex(excess_returns.index)

excess_returns = excess_returns.sub(FF_factors["RF"], axis=0)
excess_returns.head()

0              Small                                  2                              ...     4                              Big                          
1                Low      2      3      4   High    Low      2      3      4   High  ...   Low     2      3      4   High   Low     2     3      4   High
(Size, BE/ME)                                                                        ...                                                                 
1936-01-01     26.93  15.48  21.81  14.55  33.27  16.42  12.48   9.48  10.26  26.80  ...  2.12  6.86   8.91   8.34   6.16  2.54  6.35  8.61  14.83  16.28
1936-02-01      9.45  12.77   6.76  10.01   8.99   1.69   6.70   5.60   8.91   3.86  ...  2.58  4.03   6.13   4.79   8.42  1.87  1.17  3.98   3.37   3.74
1936-03-01      9.44   1.36   5.54   2.99   1.22  -0.39   1.33   3.31  -1.13   1.23  ... -0.48 -0.42   3.17   1.50  -4.38  3.56  1.25 -1.84   1.24  -3.58
1936-04-01    -28.62 -29.07 -12.39 -14.05 -21.74 -19.43 -13.37 -15.95 -16.45 -17.71  ... -8.33 -9.02 -11.92 -12.11 -12.48 -6.19 -8.28 -7.07 -10.59  -8.02
1936-05-01      1.79   8.60   1.87  10.58   5.88   5.19   4.57   7.55   5.82   6.95  ...  5.10  1.77   4.52   5.96   8.16  4.76  5.18  4.69   6.23   8.37

[5 rows x 25 columns]

---

## Fama-Mechbath Linear Regression
Function to run Fama-Macbeth linear regression, the implementation of the function is to avoid repetation of code for different data sets. The Inputs should be columns (dataframe type) of target varaibles and explanatory variable 

In [24]:
def Fama_Macbeth(independent_df, target_df):

    # time-series regressions
    X = sm.add_constant(independent_df)

    betas = []
    r_squa = []
    const = []

    for j in range(len(target_df.columns)):
        y = target_df.iloc[:, j]
        model = sm.OLS(y, X).fit()

        r_squa.append(model.rsquared)
        params = model.params

        const.append(params.iloc[0])
        betas.append(params.iloc[1:])       

    betas = pd.DataFrame(betas, index=target_df.columns)
    betas.columns = independent_df.columns

    # cross-sectional regressions
    gammas = []

    for t in range(len(target_df)):
        # 25 portfolio returns at time t
        y_t = target_df.iloc[t, :]          
        # betas are fixed across time
        X_cs = sm.add_constant(betas)       

        model_cs = sm.OLS(y_t, X_cs).fit()
        gammas.append(model_cs.params)

    gammas = pd.DataFrame(gammas, index=target_df.index)

    # risk premia estimates
    gamma_mean = gammas.mean()
    gamma_std = gammas.std()

    T = len(gammas)

    gamma_se = gamma_std / np.sqrt(T)
    gamma_t = gamma_mean / gamma_se

    results = pd.DataFrame({'Mean Gamma': gamma_mean, 'Std Dev': gamma_std, 'Std Error': gamma_se, 't-stat': gamma_t})

    return results, betas, gammas, r_squa

---

## Fama-French 3 Factor Model

$$R(t)-RF(t)=a+b[RM(t)-RF(t)]+sSMB(t)+hHML(t)+c(t)$$

In [25]:
X = FF_factors[["Mkt-RF", "SMB", "HML"]]
Y = excess_returns.copy()

FF_model_results, FF_gamma, FF_gamma0, FF_r_squa = Fama_Macbeth(X, Y)
FF_model_results.head()

,Mean Gamma,Std Dev,Std Error,t-stat
const,1.701787,8.350195,0.283098,6.011296
Mkt-RF,-1.020456,9.243081,0.313370,-3.256397
SMB,0.112178,3.105784,0.105296,1.065361
HML,0.447678,2.981375,0.101078,4.429034


---

## Macroeconomics Factor Model

In [26]:
X = macro_factors[["Div_growth", "DEF = LT Corp. - LT govt.", "TERM = ST govt.-LT govt."]]
Y = excess_returns.copy()

macro_model_results, macro_gamma, macro_gamma0, macro_r_squa = Fama_Macbeth(X, Y)
macro_model_results.copy()

,Mean Gamma,Std Dev,Std Error,t-stat
const,-0.464905,8.904348,0.301886,-1.540003
Div_growth,-0.000712,0.012236,0.000415,-1.715868
DEF = LT Corp. - LT govt.,0.002327,0.072539,0.002459,0.946242
TERM = ST govt.-LT govt.,0.016285,0.186732,0.006331,2.572388


---

## In-sample Principle Component Factors

In [27]:
# excess returns
excess_returns = df.copy()

macro_factors = macro_factors.reindex(excess_returns.index)
excess_returns = excess_returns.sub(macro_factors["TERM = ST govt.-LT govt."], axis=0)
excess_returns.head()

# covariance matrix of excess returns
cova_matrix = excess_returns.cov()

# Eigendecomposition
eigenvalues, eigenvectors = np.linalg.eigh(cova_matrix)

# eigh returns in ASCENDING order — reverse to get largest first
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx]
eigenvectors = eigenvectors[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W = eigenvectors[:, :3]

# factor time-series (T x 3)
F = excess_returns.values @ W 

in_sample_factors_df = pd.DataFrame(F, index=excess_returns.index, columns=['In-sample-PC1', 'In-sample-PC2', 'In-sample-PC3'])

in_sample_factors_df.head()

,In-sample-PC1,In-sample-PC2,In-sample-PC3
"(Size, BE/ME)",,,
1936-01-01,68.032285,-8.331803,20.995633
1936-02-01,27.507629,-1.799596,7.564300
1936-03-01,8.558362,-6.086865,-1.359546
1936-04-01,-70.181755,11.244138,-4.735477
1936-05-01,27.877746,7.509749,4.294490


### Linear regression for In-sample PC factors

In [28]:
X = in_sample_factors_df[["In-sample-PC1", "In-sample-PC2", "In-sample-PC3"]]
Y = excess_returns.copy()

insample_model_results, insample_gamma, insample_gamma0, insample_r_squa = Fama_Macbeth(X, Y)
insample_model_results.copy()

,Mean Gamma,Std Dev,Std Error,t-stat
const,1.106623,7.038259,0.238619,4.637608
In-sample-PC1,0.384713,44.541476,1.510097,0.254760
In-sample-PC2,0.262273,8.841580,0.299758,0.874951
In-sample-PC3,0.844050,6.400761,0.217006,3.889519


---

## Out-sample principle Component Factors

### Even and Odd Monthly Returns dataframes

In [29]:
# making odd monthly return and even monthly returns dataframe
odd_returns_df = df.copy()
odd_returns_df = odd_returns_df[::1]

even_returns_df = df.copy()
even_returns_df = even_returns_df[::1]

# covariance matrix
cova_matrix_even = even_returns_df.cov()
cova_matrix_odd = odd_returns_df.cov()

In [30]:
# Eigendecomposition for even
eigenvalues_even, eigenvectors_even = np.linalg.eigh(cova_matrix_even)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_even)[::-1]
eigenvalues_even  = eigenvalues_even[idx]
eigenvectors_even = eigenvectors_even[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_even = eigenvectors_even[:, :3]

# Eigendecomposition for odd
eigenvalues_odd, eigenvectors_odd = np.linalg.eigh(cova_matrix_odd)

# eigh returns in ascending order — reverse to get largest first
idx = np.argsort(eigenvalues_odd)[::-1]
eigenvalues_odd  = eigenvalues_odd[idx]
eigenvectors_odd = eigenvectors_odd[:, idx]    # 25 × 25, columns are eigenvectors

# extracting 3 largest eigenvectors, the factor weights is 25 x 3
W_odd = eigenvectors_odd[:, :3]

# factor time-series (T x 25)
F_even = even_returns_df.values @ W_odd
F_odd = odd_returns_df.values @ W_even

# Create empty dataframe with full index
out_sample_factors_df = pd.DataFrame(index=df.index, columns=['Out-sample-PC1', 'Out-sample-PC2', 'Out-sample-PC3'], dtype=float)

# Assign values to correct timestamps
out_sample_factors_df.loc[even_returns_df.index] = F_even
out_sample_factors_df.loc[odd_returns_df.index]  = F_odd

out_sample_factors_df.head()

,Out-sample-PC1,Out-sample-PC2,Out-sample-PC3
"(Size, BE/ME)",,,
1936-01-01,68.053580,-8.384532,20.986663
1936-02-01,27.545814,-1.812116,7.559540
1936-03-01,8.608734,-6.075599,-1.375887
1936-04-01,-70.161977,11.272641,-4.723863
1936-05-01,27.898146,7.497325,4.310874


### Linear regression for Out-sample PC factors

In [31]:
X = out_sample_factors_df[["Out-sample-PC1", "Out-sample-PC2", "Out-sample-PC3"]]
Y = excess_returns.copy()

outsample_model_results, outsample_gamma, outsample_gamma0, outsample_r_squa = Fama_Macbeth(X, Y)
outsample_model_results.copy()

,Mean Gamma,Std Dev,Std Error,t-stat
const,1.106406,7.040335,0.238690,4.635332
Out-sample-PC1,0.384855,44.593567,1.511863,0.254557
Out-sample-PC2,0.260631,8.848097,0.299979,0.868834
Out-sample-PC3,0.844566,6.397054,0.216880,3.894154


----

### Simplified Fama-Macbeth Process

In [32]:
def Fama_Macbeth_gamma(independent_df, target_df):

    # time-series regressions to estimate betas
    X = sm.add_constant(independent_df)

    betas = []
    r_squa = []
    alphas = []

    for j in range(len(target_df.columns)):
        y = target_df.iloc[:, j]
        model = sm.OLS(y, X).fit()

        r_squa.append(model.rsquared)
        params = model.params

        alphas.append(params.iloc[0])     
        betas.append(params.iloc[1:])      

    betas = pd.DataFrame(betas, index=target_df.columns)
    betas.columns = independent_df.columns

    # cross-sectional regression to estimate gamma
    avg_returns = target_df.mean()

    X_cs = sm.add_constant(betas)
    gamma_model = sm.OLS(avg_returns, X_cs).fit()

    gamma = gamma_model.params

    # standard errors (optional)
    gamma_se = gamma_model.bse
    gamma_t = gamma_model.tvalues

    results = pd.DataFrame({'Gamma': gamma, 'Std Error': gamma_se, 't-stat': gamma_t})

    return results, betas, alphas, r_squa